# TP3 - Problem Settings and Data Generation

## Temperature monitoring on the 3D rock

This notebook defines the **problem settings** for **Test Problem 3 (TP3)**, which addresses a **Heat Equation problem** on the **3D rock geometry**.

The purpose of this notebook is to:

*   define the physical problem and its parameters,

*   generate **simulated IoT-like boundary measurements**,

*   acquire and preprocess the 3D geometry,

*   produce the mesh and visualization-ready files required by the subsequent stage.

This notebook represents the **first stage of the TP3 pipeline** and prepares all the assets used in the **Direct Problem Submodule**.

In [ ]:
import sys
from pyprojroot import here

PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

In [ ]:
from paths import DRT_PATH, TP3_PATH, MODEL_PATH, ROCK_PATH

ABS_PATH = PROJECT_ROOT + DRT_PATH + TP3_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + ROCK_PATH

model_name = "Rock1.blend"

In [ ]:
# Create needed folders
import os

lst_folders = ["figures", "files", "models"]

for name in lst_folders:
    os.makedirs(os.path.join(ABS_PATH, name), exist_ok=True)

## Libraries and Dependencies

We start by importing all the libraries required for:
- geometry handling and mesh generation,
- data management,
- preparation of visualization files.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modelaquisition.bl2pina import Blend2Pina
from modelaquisition.bl2msh import Blend2Mesh
from modelaquisition.msh2xdmf import Msh2Xdmf

## Numerical Precision

Double precision is enforced

In [ ]:
torch.set_default_dtype(torch.float64)

## Physical Problem: the Heat Equation

We consider a 3D domain $\Omega \subseteq \mathbb{R}^3$ representing a **rock geometry**, with boundary $\Gamma = \partial \Omega$.  
The physical phenomenon is described by the following differenzial problem:

\begin{cases}
    u_t (x, y, z, t ) - \Delta u (x, y, z, t ) = 0 & \Omega \times [0, 1]
    \tag{1}
\end{cases}
with boundary conditions described by data generated below and initial condition that fixes a a constant temperature of the domain.

# Generation of Data for boundary conditions

In this stage, we generate **simulated measurements** that emulate IoT sensor data.

Sensors are assumed to be located on the boundary of the 3D domain.

To guarantee reproducibility, the random seed is fixed. The temperature values are generated from a normal distribution with mean $m=22.4$ and standard deviation $\sigma=1.2$.

In [ ]:
seed = 10
np.random.seed(seed)

In [ ]:
number_samples = 100
mean_temperature = 22.4
std_temperature = 1.2

temperature = np.random.normal(
    loc=mean_temperature,
    scale=std_temperature,
    size=(number_samples)
)

The next block identifies temperature values that lie outside the interval defined by $[m - 2\sigma, m + 2\sigma]$, where $m$ is the mean temperature and $\sigma$ is the standard deviation. Values exceeding the upper bound are clipped to $m + 2\sigma$, while values below the lower bound are clipped to $m - 2\sigma$.
This operation limits the influence of extreme values while preserving the overall statistical structure of the data.

In [ ]:
over_std_idx = np.logical_and(abs(temperature - mean_temperature) - 2*std_temperature >= 0, temperature - mean_temperature >= 0)
under_std_idx = np.logical_and(abs(temperature - mean_temperature) - 2*std_temperature >= 0, temperature - mean_temperature <= 0)
temperature[over_std_idx] = mean_temperature + 2*std_temperature
temperature[under_std_idx] = mean_temperature - 2*std_temperature

The temperature array is split into two halves. The first half is sorted in ascending order, while the second half is sorted in ascending order and then reversed to obtain a descending trend. Finally, the two parts are concatenated to form a single sequence that increases in the first portion and decreases in the second.

In [ ]:
half = int(number_samples/2)

first_part = temperature[:half]
first_part.sort()
second_part = temperature[half:]
second_part.sort()
second_part = np.flip(second_part)

temperature = np.concatenate([first_part, second_part])

A uniformly spaced datetime index is generated. The resulting timestamps are combined with the temperature values to build a pandas DataFrame with two columns: `Datetime` and `Temperature_C`.

In [ ]:
date_range = pd.date_range(start='2024-01-01 00:00:00', end='2024-01-01 23:59:00', periods=number_samples)
temperature_data = pd.DataFrame({'Datetime': date_range, 'Temperature_C': temperature})

Plotting of simulated temperature data and saving.

In [ ]:
mean_plus_std = mean_temperature + std_temperature
mean_minus_std = mean_temperature - std_temperature

plt.figure(figsize=(20, 6))
plt.scatter(temperature_data['Datetime'], temperature_data['Temperature_C'], color='blue', marker="p")
plt.axhline(mean_temperature, color='red', linestyle='--', label='Mean', linewidth=2.2)
plt.axhline(mean_plus_std, color='orange', linestyle='-.', label='Mean + Std Dev', linewidth=2.2)
plt.axhline(mean_minus_std, color='orange', linestyle=':', label='Mean - Std Dev', linewidth=2.2)
plt.legend()
plt.title("Sampled data related to surface temperature")
plt.xlabel("time")
plt.ylabel("Temperature  (°C)")
plt.grid(True)
plt.xticks(temperature_data['Datetime'], temperature_data['Datetime'].dt.strftime('%H:%M'), rotation=45)
plt.tight_layout()
plt.savefig(ABS_PATH+"figures/weather_data.png", transparent=True, dpi=300)
plt.show()

In [ ]:
temperature_data.to_csv(ABS_PATH+"files/tempdata.csv", sep=";", index=False)

Connection of temperature data with boundary points

In [ ]:
num_points_b = 100

rock = Blend2Pina(LOAD_MODEL + model_name)
surface = rock.boundary()
points_boundary = surface.sample(num_points_b)

LT_lst = list()
time = np.linspace(0, 1, len(temperature_data))
T_boundary = list()

for i in range(1, time.shape[0]):
  for sp_el in points_boundary:
    LT_lst.append([sp_el.tensor[0].item(), sp_el.tensor[1].item(), sp_el.tensor[2].item(), time[i], temperature_data["Temperature_C"][i]])

data_boundary = pd.DataFrame(LT_lst, columns=["x", "y", "z", "t", "u"])
data_boundary.to_csv(ABS_PATH+"files/data.csv", sep=";", index=None)

## Generation of collocation points

In [ ]:
num_points_int = 10_000
domain = rock.intern()
points_internal = domain.sample(num_points_int)

In [ ]:
df_int = pd.DataFrame(
    points_internal.tensor.detach().numpy()
)
df_int.to_csv("./files/input_int.csv", sep = ";")

## Mesh Generation and Visualization Files

The 3D rock geometry is discretized to generate a computational mesh suitable for numerical simulations.

In addition:
- visualization-ready files (.xdmf and associated data) are produced,
- these files enable inspection of solutions and errors in ParaView,
- the same mesh is reused consistently across all TP1 stages.

In [ ]:
rock_msh = Blend2Mesh(LOAD_MODEL + model_name, "rock")

In [ ]:
rock_msh.create_mesh(len_msh=0.07)

In [ ]:
rock_xdmf = Msh2Xdmf("rock.msh", "rock")
rock_xdmf.to_xdmf()